In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(
    "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/subjects_AND_sampling_metadata.csv",
    parse_dates=["created_at"],
    usecols=["id", "created_at", "tweets_historical", "name", "username", "description", "location",
             "public_metrics.followers_count", "profile_image_url", "clean...pool"],  # the file has 121 columns and is 16 GB
)

In [ ]:
# filter the dataframe to only include tweets from the year leading up to the election date
election_date = pd.Timestamp("2024-11-05", tz="UTC")
start_period = election_date - pd.DateOffset(years=1)
end_period = election_date
df = df[(df["created_at"] >= start_period) & (df["created_at"] <= end_period)].copy()

# subjects_AND_sampling_metadata.csv has each tweet once per row of the user's sampling
# records (0.link_x_user_data.R merges the pool and tweet history on user id, with
# allow.cartesian=TRUE), so keep one copy of each tweet
n_before = len(df)
df = df.drop_duplicates(subset=["id", "created_at", "tweets_historical"]).copy()
print(f"Dropped {n_before - len(df):,} duplicate tweet rows, {len(df):,} left")

In [ ]:
WINDOW_DAYS = 7  # variable to adjust time window size

df['created_at'] = pd.to_datetime(df['created_at'], utc=True)

# Create WINDOW_DAYS-day windows starting from first available date
first_date = df['created_at'].min()
df['days_since_start'] = (df['created_at'] - first_date).dt.days
df['window_number'] = (df['days_since_start'] // WINDOW_DAYS).astype(int)
df['time_window_start'] = first_date + pd.to_timedelta(df['window_number'] * WINDOW_DAYS, unit='D')
df['window_weeks_to_election'] = ((election_date - df['time_window_start']).dt.days // 7).astype(int)

# Build per-tweet dictionary
df['tweet_dict'] = df.apply(
    lambda row: {
        'text': row['tweets_historical'],
        'created_at': row['created_at'].tz_convert(None)
    },
    axis=1
)

# Group tweets by user and window
grouped = df.groupby(['id', 'time_window_start'])

def combine_tweets(group):
    weeks_to_election = int(group['window_weeks_to_election'].iloc[0])
    tweets_list = group['tweet_dict'].tolist()
    tweets_list.insert(0, f"{weeks_to_election} weeks to election")
    return tweets_list

# Aggregate tweets per window
tweets_df = grouped.apply(combine_tweets).reset_index(name='tweets_struct')

# Get constant user metadata
user_meta_cols = ['id', 'name', 'username', 'description', 'location',
                  'public_metrics.followers_count', 'profile_image_url', 'clean...pool']
user_meta_df = df[user_meta_cols].drop_duplicates(subset=['id'])

# Fill missing windows for all users
all_users = df['id'].unique()
all_windows = pd.date_range(first_date,
                            df['time_window_start'].max(),
                            freq=f'{WINDOW_DAYS}D')
full_index = pd.MultiIndex.from_product([all_users, all_windows],
                                        names=['id', 'time_window_start'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge tweets per window
full_df = full_df.merge(tweets_df, on=['id', 'time_window_start'], how='left')
full_df = full_df.merge(user_meta_df, on='id', how='left')

# Fill missing tweets_struct with empty list
full_df['tweets_struct'] = full_df['tweets_struct'].apply(lambda x: x if isinstance(x, list) else [])
full_df['is_empty_window'] = full_df['tweets_struct'].apply(lambda x: len(x) == 0)

# Compute weeks to election for all windows
full_df['window_weeks_to_election'] = ((election_date - full_df['time_window_start']).dt.days // 7).astype(int)

In [ ]:
full_df

In [ ]:
df_with_desc = pd.read_csv("../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/final_user_profiles_cleaned.csv")  

# Merge on "id"
merged_df = full_df.merge(
    df_with_desc[["id", "profile_image_url", "profile_image_description", "profile_image_main_info"]],
    on="id",
    how="left",
    suffixes=("", "_new")  
)

# Overwrite profile_image_url with the new value 
merged_df["profile_image_url"] = merged_df["profile_image_url_new"].combine_first(merged_df["profile_image_url"])

# Drop the temporary column used for overwrite
merged_df = merged_df.drop(columns=["profile_image_url_new"])

merged_df.to_csv(f"../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/all_user_windows_{WINDOW_DAYS}d.csv", index=False)
merged_df

In [ ]:
# First, check what type the data actually is
print(f"Type of tweets_struct: {type(full_df['tweets_struct'].iloc[100])}")

# If it's a string, check its length and first few characters
if isinstance(full_df['tweets_struct'].iloc[100], str):
    print(f"String length: {len(full_df['tweets_struct'].iloc[100])}")
    print(f"First 200 characters: {full_df['tweets_struct'].iloc[100]}")

In [ ]:
import matplotlib.pyplot as plt

num_users = merged_df["id"].nunique()
print(f"Number of unique users sampled: {num_users}")

num_windows = merged_df.groupby(["id", "time_window_start"]).ngroups
print(f"Number of sampled windows: {num_windows}")

In [ ]:
# Number of unique windows per user
windows_per_user = full_df.groupby('id')['time_window_start'].nunique()

# Compute stats
avg_windows = windows_per_user.mean()
min_windows = windows_per_user.min()
max_windows = windows_per_user.max()
std_windows = windows_per_user.std()

print(f"Average windows per user: {avg_windows:.2f}")
print(f"Min windows per user: {min_windows}")
print(f"Max windows per user: {max_windows}")
print(f"Standard deviation: {std_windows:.2f}")

In [ ]:
empty_percentage = (full_df['is_empty_window'].mean() * 100).round(2)
print(f"Percentage of empty windows: {empty_percentage}%")

In [ ]:
print(full_df.info())   
print(full_df.head())  
print(full_df.tail())